# Document conversion with SmolDocling and OpenVINO

<div style="display: flex; align-items: center;">
<img src="https://github.com/user-attachments/assets/52a86ee7-065c-4ca5-b0b3-2458a79a6c9d" alt="SmolDocling" style="width: 200px; height: auto; margin-right: 20px;">
<div>
        <p>SmolDocling is a multimodal Image-Text-to-Text model designed for efficient document conversion. It retains Docling's most popular features while ensuring full compatibility with <a href=https://github.com/docling-project/docling> Docling </a> through seamless support for <strong>DoclingDocuments</strong> (The JSON that can be exported using Docling follows this schema. The DoclingDocument type also models the metadata that may be attached to the converted document.)
        </p>
</div>
</div>

🚀 Model Features:  
- 🏷️ **DocTags for Efficient Tokenization** – Introduces DocTags an efficient and minimal representation for documents that is fully compatible with **DoclingDocuments**.  
- 🔍 **OCR (Optical Character Recognition)** – Extracts text accurately from images.  
- 📐 **Layout and Localization** – Preserves document structure and document element **bounding boxes**.  
- 💻 **Code Recognition** – Detects and formats code blocks including identation.  
- 🔢 **Formula Recognition** – Identifies and processes mathematical expressions.  
- 📊 **Chart Recognition** – Extracts and interprets chart data.  
- 📑 **Table Recognition** – Supports column and row headers for structured table extraction.  
- 🖼️ **Figure Classification** – Differentiates figures and graphical elements.  
- 📝 **Caption Correspondence** – Links captions to relevant images and figures.  
- 📜 **List Grouping** – Organizes and structures list elements correctly.  
- 📄 **Full-Page Conversion** – Processes entire pages for comprehensive document conversion including all page elements (code, equations, tables, charts etc.) 
- 🔲 **OCR with Bounding Boxes** – OCR regions using a bounding box.
- 📂 **General Document Processing** – Trained for both scientific and non-scientific documents.  
- 🔄 **Seamless Docling Integration** – Import into **Docling** and export in multiple formats.

More details about model can be found in the [model card](https://huggingface.co/ds4sd/SmolDocling-256M-preview) and [paper](https://huggingface.co/papers/2503.11576).

In this tutorial we consider how to convert and run SmolDocling model using OpenVINO [Optimum Intel](https://github.com/huggingface/optimum-intel) integration. Additionally, we demonstrate how to apply model optimization techniques like weights compression using [NNCF](https://github.com/openvinotoolkit/nncf).

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Convert and Optimize model](#Convert-and-Optimize-model)
    - [Compress model weights](#Compress-model-weights)
- [Run model inference](#Run-model-inference)
    - [Supported Instructions](#Supported-Instructions)
    - [Select Inference Device](#Select-Inference-Device)
    - [Run model to generate DocTags](#Run-model-to-generate-DocTags)
- [Interactive demo](#Interactive-demo)


### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/smoldocling/smoldocling.ipynb" />


## Prerequisites
[back to top ⬆️](#Table-of-contents:)

In [1]:
%pip uninstall -q -y optimum optimum-intel optimum-onnx
%pip install -q "torch==2.8" "torchvision==0.23.0" "Pillow" "gradio>=4.36" "opencv-python" "docling_core" "num2words" --extra-index-url https://download.pytorch.org/whl/cpu
%pip install  -q -U "openvino>=2025.0.0" "openvino-tokenizers>=2025.0.0" "nncf>=2.15.0"
%pip install -q "https://codeload.github.com/huggingface/optimum-intel/tar.gz/HEAD" "transformers==4.57.6" "diffusers>=0.29.0,<0.38.0" --extra-index-url https://download.pytorch.org/whl/cpu

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tokenizers 0.22.2 requires huggingface-hub<2.0,>=0.16.4, but you have huggingface-hub 2.1.1 which is incompatible.
diffusers 0.37.1 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 2.1.1 which is incompatible.
transformers 4.57.6 requires huggingface-hub<1.0,>=0.34.0, but you have huggingface-hub 2.1.1 which is incompatible.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.29.1 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


Note: you may need to restart the kernel to use updated packages.

In [2]:
from pathlib import Path
import requests

if not Path("cmd_helper.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/cmd_helper.py")
    open("cmd_helper.py", "w").write(r.text)

if not Path("notebook_utils.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py")
    open("notebook_utils.py", "w").write(r.text)

# Convert and Optimize model
[back to top ⬆️](#Table-of-contents:)

SmolDocling is PyTorch model. OpenVINO supports PyTorch models via conversion to OpenVINO Intermediate Representation (IR). [OpenVINO model conversion API](https://docs.openvino.ai/2025/openvino-workflow/model-preparation.html#convert-a-model-with-python-convert-model) should be used for these purposes. `ov.convert_model` function accepts original PyTorch model instance and example input for tracing and returns `ov.Model` representing this model in OpenVINO framework. Converted model can be used for saving on disk using `ov.save_model` function or directly loading on device using `core.compile_model`. 

For convenience, we will use OpenVINO integration with HuggingFace Optimum. 🤗 [Optimum Intel](https://huggingface.co/docs/optimum/intel/index) is the interface between the 🤗 Transformers and Diffusers libraries and the different tools and libraries provided by Intel to accelerate end-to-end pipelines on Intel architectures.

Among other use cases, Optimum Intel provides a simple interface to optimize your Transformers and Diffusers models, convert them to the OpenVINO Intermediate Representation (IR) format and run inference using OpenVINO Runtime. `optimum-cli` provides command line interface for model conversion and optimization. 

General command format:

```bash
optimum-cli export openvino --model <model_id_or_path> --task <task> <output_dir>
```

where task is task to export the model for, if not specified, the task will be auto-inferred based on the model. You can find a mapping between tasks and model classes in Optimum TaskManager [documentation](https://huggingface.co/docs/optimum/exporters/task_manager). Additionally, you can specify weights compression using `--weight-format` argument with one of following options: `fp32`, `fp16`, `int8` and `int4`. Fro int8 and int4 [nncf](https://github.com/openvinotoolkit/nncf) will be used for  weight compression. More details about model export provided in [Optimum Intel documentation](https://huggingface.co/docs/optimum/intel/openvino/export#export-your-model).


### Compress model weights
[back to top ⬆️](#Table-of-contents:)
For reducing memory consumption, weights compression optimization can be applied using [NNCF](https://github.com/openvinotoolkit/nncf). 

<details>
    <summary><b>Click here for more details about weight compression</b></summary>
Weight compression aims to reduce the memory footprint of a model. It can also lead to significant performance improvement for large memory-bound models, such as Large Language Models (LLMs). LLMs and other models, which require extensive memory to store the weights during inference, can benefit from weight compression in the following ways:

* enabling the inference of exceptionally large models that cannot be accommodated in the memory of the device;

* improving the inference performance of the models by reducing the latency of the memory access when computing the operations with weights, for example, Linear layers.

[Neural Network Compression Framework (NNCF)](https://github.com/openvinotoolkit/nncf) provides 4-bit / 8-bit mixed weight quantization as a compression method primarily designed to optimize LLMs. The main difference between weights compression and full model quantization (post-training quantization) is that activations remain floating-point in the case of weights compression which leads to a better accuracy. Weight compression for LLMs provides a solid inference performance improvement which is on par with the performance of the full model quantization. In addition, weight compression is data-free and does not require a calibration dataset, making it easy to use.

`nncf.compress_weights` function can be used for performing weights compression. The function accepts an OpenVINO model and other compression parameters. Compared to INT8 compression, INT4 compression improves performance even more, but introduces a minor drop in prediction quality.

More details about weights compression, can be found in [OpenVINO documentation](https://docs.openvino.ai/2025/openvino-workflow/model-optimization-guide/weight-compression.html).
</details>

In [3]:
import ipywidgets as widgets

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("smoldocling.ipynb")

pt_model_id = "ds4sd/SmolDocling-256M-preview"
model_dir = Path(pt_model_id.split("/")[-1])

to_compress = widgets.Checkbox(value=True, description="Compress model")
to_compress

Checkbox(value=True, description='Compress model')

In [4]:
from cmd_helper import optimum_cli

if to_compress.value:
    model_export_dir = model_dir / ("INT8" if to_compress.value else "FP16")

if not model_export_dir.exists():
    additional_args = {"weight-format": "int8"} if to_compress.value else {"weight-format": "fp16"}
    optimum_cli(pt_model_id, model_export_dir, additional_args=additional_args)

## Run model inference
[back to top ⬆️](#Table-of-contents:)

OpenVINO integration with Optimum Intel provides ready-to-use API for model inference that can be used for smooth integration with transformers-based solutions. For loading model, we will use `OVModelForVisualCausalLM` class that have compatible interface with Transformers Gemma3 implementation. For loading a model, `from_pretrained` method should be used. It accepts path to the model directory or model_id from HuggingFace hub (if model is not converted to OpenVINO format, conversion will be triggered automatically). Additionally, we can provide an inference device, quantization config (if model has not been quantized yet) and device-specific OpenVINO Runtime configuration. More details about model inference with Optimum Intel can be found in [documentation](https://huggingface.co/docs/optimum/intel/openvino/inference).

### Supported Instructions
[back to top ⬆️](#Table-of-contents:)


Model supports following list of preformatted commands that describes which action is required to be performed:

<table>
  <tr>
    <td><b>Description</b></td>
    <td><b>Instruction</b></td>
    <td><b>Comment</b></td>
  </tr>
  <tr>
    <td><b>Full conversion</b></td>
    <td>Convert this page to docling.</td>
    <td>DocTags representation</td>
  </tr>
  <tr>
    <td><b>Chart</b></td>
    <td>Convert chart to table.</td>
    <td>(e.g., &lt;chart&gt;)</td>
  </tr>
  <tr>
    <td><b>Formula</b></td>
    <td>Convert formula to LaTeX.</td>
    <td>(e.g., &lt;formula&gt;)</td>
  </tr>
  <tr>
    <td><b>Code</b></td>
    <td>Convert code to text.</td>
    <td>(e.g., &lt;code&gt;)</td>
  </tr>
  <tr>
    <td><b>Table</b></td>
    <td>Convert table to OTSL.</td>
    <td>(e.g., &lt;otsl&gt;) OTSL: <a href="https://arxiv.org/pdf/2305.03393">Lysak et al., 2023</a></td>
  </tr>
  <tr>
    <td rowspan=4><b>Actions and Pipelines</b></td>
    <td>OCR the text in a specific location: &lt;loc_155&gt;&lt;loc_233&gt;&lt;loc_206&gt;&lt;loc_237&gt;</td>
    <td></td>
  </tr>
  <tr>
    <td>Identify element at: &lt;loc_247&gt;&lt;loc_482&gt;&lt;10c_252&gt;&lt;loc_486&gt;</td>
    <td></td>
  </tr>
  <tr>
    <td>Find all 'text' elements on the page, retrieve all section headers.</td>
    <td></td>
  </tr>
  <tr>
    <td>Detect footer elements on the page.</td>
    <td></td>
  </tr>
</table>

In [5]:
from optimum.intel.openvino import OVModelForVisualCausalLM
from transformers import AutoProcessor, TextStreamer

Multiple distributions found for package optimum. Picked distribution: optimum


### Select Inference Device
[back to top ⬆️](#Table-of-contents:)

In [6]:
from notebook_utils import device_widget

device = device_widget(default="AUTO", exclude=["NPU"])

device

Dropdown(description='Device:', index=2, options=('CPU', 'GPU', 'AUTO'), value='AUTO')

In [7]:
model = OVModelForVisualCausalLM.from_pretrained(model_export_dir, device=device.value)
processor = AutoProcessor.from_pretrained(model_export_dir)

[21:38:16.9490]D[plugin.cpp:289][AUTO] deviceNameWithID:CPU, defaultDeviceID:, uniqueName:CPU_
[21:38:16.9491]I[plugin.cpp:479][AUTO] device:CPU, config:LOG_LEVEL=LOG_NONE
[21:38:16.9491]I[plugin.cpp:479][AUTO] device:CPU, config:PERFORMANCE_HINT=LATENCY
[21:38:16.9491]I[plugin.cpp:479][AUTO] device:CPU, config:PERFORMANCE_HINT_NUM_REQUESTS=0
[21:38:16.9491]I[plugin.cpp:479][AUTO] device:CPU, config:PERF_COUNT=NO
[21:38:16.9491]I[plugin.cpp:484][AUTO] device:CPU, priority:0
[21:38:16.9491]I[plugin.cpp:510][AUTO] Original device list() size to load the model: 0
[21:38:16.9491]I[schedule.cpp:17][AUTO] scheduler starting
[21:38:16.9491]I[auto_schedule.cpp:163][AUTO] select device:CPU
[21:38:16.9587]I[auto_schedule.cpp:284][AUTO] Only one device or multiple devices of the same type will use passthrough compiled model


  PERFORMANCE_HINT: PerformanceMode.LATENCY


  OPTIMAL_NUMBER_OF_INFER_REQUESTS: 1


  NETWORK_NAME: Model6


  CPU:


...

  LOADED_FROM_CACHE: False


  PERF_COUNT: False


EXECUTION_DEVICES:


  CPU: AMD RYZEN AI MAX+ PRO 395 w/ Radeon 8060S      


[21:38:17.4373]I[auto_schedule.cpp:341][AUTO] Device: [CPU]: Compile model took 478.567848 ms
[21:38:17.4380]I[auto_schedule.cpp:118][AUTO] device:CPU compiling model finished
[21:38:17.4381]D[auto_schedule.cpp:124][AUTO] device:CPU, GetConfig:SUPPORTED_PROPERTIES=SUPPORTED_PROPERTIES NETWORK_NAME OPTIMAL_NUMBER_OF_INFER_REQUESTS NUM_STREAMS INFERENCE_NUM_THREADS PERF_COUNT INFERENCE_PRECISION_HINT PERFORMANCE_HINT EXECUTION_MODE_HINT PERFORMANCE_HINT_NUM_REQUESTS ENABLE_
[21:38:17.4381]D[auto_schedule.cpp:124][AUTO] device:CPU, GetConfig:NETWORK_NAME=Model6
[21:38:17.4381]D[auto_schedule.cpp:124][AUTO] device:CPU, GetConfig:OPTIMAL_NUMBER_OF_INFER_REQUESTS=1
[21:38:17.4381]D[auto_schedule.cpp:124][AUTO] device:CPU, GetConfig:NUM_STREAMS=1
[21:38:17.4381]D[auto_schedule.cpp:124][AUTO] device:CPU, GetConfig:INFERENCE_NUM_THREADS=16
[21:38:17.4381]D[auto_schedule.cpp:124][AUTO] device:CPU, GetConfig:PERF_COUNT=NO
[21:38:17.4381]D[auto_schedule.cpp:124][AUTO] device:CPU, GetConfig:INFEREN

### Run model to generate DocTags
[back to top ⬆️](#Table-of-contents:)


DocTags create a clear and structured system of tags and rules that separate text from the document's structure. This makes things easier for Image-to-Sequence models by reducing confusion. On the other hand, converting directly to formats like HTML or Markdown can be messy — it often loses details, doesn’t clearly show the document’s layout, and increases the number of tokens, making processing less efficient. DocTags are integrated with Docling, which allows export to HTML, Markdown, and JSON.

![doctags.png](https://github.com/user-attachments/assets/de2b4db8-cfae-4dd1-a38b-bc778f78f2a7)

In [8]:
from docling_core.types.doc import DoclingDocument
from docling_core.types.doc.document import DocTagsDocument
from transformers.image_utils import load_image
from IPython.display import Markdown

test_image_path = Path("ov_docs.png")
test_image_url = "https://github.com/openvinotoolkit/openvino_notebooks/assets/29454499/aa46ef0c-c14d-4bab-8bb7-3b22fe73f6bc"
if not test_image_path.exists():
    image = load_image(test_image_url)
    image.save(test_image_path)
else:
    image = load_image(str(test_image_path))

display(image)

messages = [
    {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": "Convert this page to docling."}]},
]

# Prepare inputs
prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(text=prompt, images=[image], return_tensors="pt")

print("\n\n\nRAW OUTPUT:")
generated_ids = model.generate(**inputs, max_new_tokens=8192, streamer=TextStreamer(processor, skip_special_tokens=True, skip_prompt=True))
prompt_length = inputs.input_ids.shape[1]
trimmed_generated_ids = generated_ids[:, prompt_length:]
doctags = processor.batch_decode(
    trimmed_generated_ids,
    skip_special_tokens=False,
)[0].lstrip()

# Populate document
doctags_doc = DocTagsDocument.from_doctags_and_image_pairs([doctags], [image])
# create a docling document
doc = DoclingDocument(name="Document")
doc.load_from_doctags(doctags_doc)

print("\n\nMarkdown result")
display(Markdown(doc.export_to_markdown()))

ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

## Interactive demo
[back to top ⬆️](#Table-of-contents:)

In [9]:
if not Path("gradio_helper.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/notebooks/smoldocling/gradio_helper.py")
    open("gradio_helper.py", "w").write(r.text)